In [0]:
# define the catalog and database
catalog = 'uk_property_insights'
database = 'default'
bronze_table = 'bronze_property_prices'

# set the catalog and database
spark.catalog.setCurrentCatalog(catalog)
spark.catalog.setCurrentDatabase(database)
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current database: {spark.catalog.currentDatabase()}")

# check if the table exists
table_exists = spark.catalog.tableExists(bronze_table)

# exit the notebook if the table does not exist
if not table_exists:
    raise Exception(f"Table {bronze_table} does not exist.")

In [0]:
# Get row count before processing new data
count_query = f"select count(*) from {bronze_table};"
try:
    row_count_before = spark.sql(count_query).collect()[0][0]
    print("Bronze table row count:", row_count_before)
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" not in str(e):
        raise
    else:
        print(f"Table {bronze_table} does not exist.")
        table_exists = False


In [0]:
# List CSV files in landing zone 
base_path = f"/Volumes/{catalog}/landing/property_prices"
new_files = base_path + "/new"
csv_files = [f.name for f in dbutils.fs.ls(new_files) if f.name.endswith('.csv')]
print("List of new CSV files to process:\n", csv_files)

In [0]:
# Copy files from the parent folder to the new folder
from datetime import datetime

source_folder = "dbfs:/Volumes/uk_property_insights/landing/property_prices/"
target_folder = "dbfs:/Volumes/uk_property_insights/landing/property_prices/new/"
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
for f in dbutils.fs.ls(source_folder):
    if f.isFile():
        name_parts = f.name.rsplit('.', 1)
        if len(name_parts) == 2:
            new_name = f"{name_parts[0]}-{timestamp}.{name_parts[1]}"
        else:
            new_name = f"{f.name}-{timestamp}"
        dbutils.fs.cp(f.path, f"{target_folder}{new_name}")

In [0]:
# Process new files
from pyspark.sql.functions import current_timestamp

# Define column names for UK property price data
column_names = [
    "transaction_id",
    "price",
    "transfer_date",
    "postcode",
    "property_type",
    "new_build",
    "tenure",
    "primary_addressable_object_name",
    "secondary_addressable_object_name",
    "street",
    "locality",
    "town_city",
    "district",
    "county",
    "ppd_category_type",
    "record_status"
]

ingestion_time = current_timestamp()
rows_processed = 0

for file in csv_files:
    # Read file
    path = f"{new_files}/{file}"
    print("Processing file:", path)
    df = spark.read.csv(path, header=False, inferSchema=True)

    # Check number of columns
    if len(df.columns) != len(column_names):
        # Move file to failed folder
        print(f"File {file} has {len(df.columns)} columns, expected {len(column_names)} columns.")
        failed_path = f"{base_path}/failed/{file}"
        dbutils.fs.mv(path, failed_path)
        continue
    
    # Rename columns
    df = df.toDF(*column_names)

    # Add ingestion timestamp
    df = df.withColumn("ingestion_time", ingestion_time)
    
    # Write to table, expect table to exist
    try:
        df.write \
          .format("delta") \
          .mode("append") \
          .option("overwriteSchema", "false") \
          .saveAsTable(bronze_table)
    except Exception as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
            print(f"Error: Table {bronze_table} does not exist. Processing aborted.")
        raise

    # Count rows processed
    rows_processed += df.count()
    
    # Move file to archive
    print("File processed, moving to archive")
    archive_path = f"{base_path}/processed/{file}"
    dbutils.fs.mv(path, archive_path)
    
print(f"Total rows processed: {rows_processed}")

In [0]:
# Validate rows processed
row_count_after = spark.sql(count_query).collect()[0][0]
assert row_count_after == row_count_before + rows_processed
print(f"Row count validation passed")

In [0]:
%sql
select year(transfer_date), count(*) 
from bronze_property_prices
group by year(transfer_date)
order by year(transfer_date);